# QML loss via reduce_mean (QAE-style on the simulator route)

Loss = mean of per-sample expectations. After the reduce_* circuit wiring, the simulator route engages qsim BOTH at the expect block AND the reduce_mean block (QAE-style amplitude estimation).

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the engine picks the route (classical / simulator / accelerator) at preflight time.

## Background

**Problem:** mean of expectation values across a batch. **Classical:** numpy mean. **Quantum:** Quantum Amplitude Estimation — `reduce_mean`'s quantum lowering emits an rz-encoded measurement that estimates the sum/mean.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx import ops
from uniqx.core import types
from uniqx.ops import expect

n = 4
rng = np.random.default_rng(101)
M = rng.standard_normal((n, n)) * 0.2
O = 0.5 * (M + M.T)
states = rng.standard_normal(2 * n) * 0.3

@trace
def prog(observable, batch):
    e0 = expect(observable, batch)
    return ops.reduce_mean(e0, result_type=types.scalar("f64"), axis=0)

module = prog(O.tolist(), states.tolist())


## Preflight — see what routes the engine found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the engine's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the engine picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
